In [1]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

In [2]:
train_data = pd.read_csv("dataset/train_metadata.csv")
val_data = pd.read_csv("dataset/val_metadata.csv")
test_data = pd.read_csv("dataset/test_metadata.csv")

In [3]:
train_data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,label
0,HAM_0000946,ISIC_0031775,nv,follow_up,60.0,male,trunk,dataset/ham10000_images_part_2/ISIC_0031775.jpg,5
1,HAM_0006097,ISIC_0027306,mel,histo,60.0,male,chest,dataset/ham10000_images_part_1/ISIC_0027306.jpg,4
2,HAM_0004348,ISIC_0033895,nv,consensus,40.0,female,unknown,dataset/ham10000_images_part_2/ISIC_0033895.jpg,5
3,HAM_0006608,ISIC_0025491,nv,histo,60.0,male,back,dataset/ham10000_images_part_1/ISIC_0025491.jpg,5
4,HAM_0005678,ISIC_0031023,mel,histo,60.0,male,chest,dataset/ham10000_images_part_2/ISIC_0031023.jpg,4


In [4]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [5]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [6]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [7]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

In [8]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomContrast(0.1)

])

In [9]:

def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [10]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [11]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [12]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [13]:

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
densenet121_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

densenet121_model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 132,103 (516.03 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [14]:
densenet121_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [15]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [16]:
history_densenet121_model = densenet121_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5


/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 232s 1s/step - accuracy: 0.1534 - loss: 1.9912 - val_accuracy: 0.2776 - val_loss: 1.7634
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 986ms/step - accuracy: 0.2689 - loss: 1.7243 - val_accuracy: 0.4474 - val_loss: 1.5440
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 984ms/step - accuracy: 0.3431 - loss: 1.5998 - val_accuracy: 0.5173 - val_loss: 1.4377
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 985ms/step - accuracy: 0.4071 - loss: 1.5343 - val_accuracy: 0.5260 - val_loss: 1.3813
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 684s 3s/step - accuracy: 0.4330 - loss: 1.4884 - val_accuracy: 0.5652 - val_loss: 1.3075
Restoring model weights from the end of the best epoch: 5.


In [17]:
train_loss, train_accuracy = densenet121_model.evaluate(train_dataset)
val_loss, val_accuracy = densenet121_model.evaluate(val_dataset)
test_loss, test_accuracy = densenet121_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 807ms/step - accuracy: 0.5491 - loss: 1.3851
47/47 ━━━━━━━━━━━━━━━━━━━━ 37s 794ms/step - accuracy: 0.5652 - loss: 1.3075
47/47 ━━━━━━━━━━━━━━━━━━━━ 37s 793ms/step - accuracy: 0.5609 - loss: 1.3423


In [18]:
densenet121_model.save("models/densenet121_model.keras")

In [19]:

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
densenet121_model_sgd = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

densenet121_model_sgd.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 132,103 (516.03 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [20]:
densenet121_model_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_densenet121_model_sgd = densenet121_model_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 226s 993ms/step - accuracy: 0.1846 - loss: 2.3002 - val_accuracy: 0.1791 - val_loss: 1.8587
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 568s 3s/step - accuracy: 0.1579 - loss: 2.1617 - val_accuracy: 0.1591 - val_loss: 1.8899
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 982ms/step - accuracy: 0.1468 - loss: 2.1138 - val_accuracy: 0.1565 - val_loss: 1.8947
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 979ms/step - accuracy: 0.1352 - loss: 2.0796 - val_accuracy: 0.1285 - val_loss: 1.9152
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [21]:
train_loss, train_accuracy = densenet121_model_sgd.evaluate(train_dataset)
val_loss, val_accuracy = densenet121_model_sgd.evaluate(val_dataset)
test_loss, test_accuracy = densenet121_model_sgd.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 372s 2s/step - accuracy: 0.1977 - loss: 1.8481
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 805ms/step - accuracy: 0.1791 - loss: 1.8587
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 809ms/step - accuracy: 0.1776 - loss: 1.8647


In [22]:
densenet121_model_sgd.save("models/densenet121_model_sgd.keras")

In [23]:

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
densenet121_model_RMSprop= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

densenet121_model_RMSprop.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 132,103 (516.03 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [24]:
densenet121_model_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_densenet121_model_RMSprop = densenet121_model_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 224s 984ms/step - accuracy: 0.2425 - loss: 2.0095 - val_accuracy: 0.5146 - val_loss: 1.4621
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 986ms/step - accuracy: 0.3766 - loss: 1.7460 - val_accuracy: 0.5979 - val_loss: 1.3267
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 986ms/step - accuracy: 0.4444 - loss: 1.6364 - val_accuracy: 0.6172 - val_loss: 1.2356
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 218s 984ms/step - accuracy: 0.4936 - loss: 1.5596 - val_accuracy: 0.5932 - val_loss: 1.2065
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 219s 986ms/step - accuracy: 0.4973 - loss: 1.4898 - val_accuracy: 0.6332 - val_loss: 1.1030
Restoring model weights from the end of the best epoch: 5.


In [25]:
train_loss, train_accuracy = densenet121_model_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = densenet121_model_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = densenet121_model_RMSprop.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 179s 807ms/step - accuracy: 0.6061 - loss: 1.2040
47/47 ━━━━━━━━━━━━━━━━━━━━ 38s 800ms/step - accuracy: 0.6332 - loss: 1.1030
47/47 ━━━━━━━━━━━━━━━━━━━━ 37s 794ms/step - accuracy: 0.6241 - loss: 1.1279


In [26]:
densenet121_model_RMSprop.save("models/densenet121_model_RMSprop.keras")

In [27]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [28]:

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
densenet121_model_64= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

densenet121_model_64.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 132,103 (516.03 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [29]:
densenet121_model_64.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_densenet121_model_64 = densenet121_model_64.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.2986 - loss: 1.7419 - val_accuracy: 0.5047 - val_loss: 1.3627
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 219s 2s/step - accuracy: 0.4755 - loss: 1.4795 - val_accuracy: 0.5173 - val_loss: 1.2946
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 221s 2s/step - accuracy: 0.5184 - loss: 1.3605 - val_accuracy: 0.5453 - val_loss: 1.1880
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.5275 - loss: 1.2763 - val_accuracy: 0.5113 - val_loss: 1.2451
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.5545 - loss: 1.2509 - val_accuracy: 0.6398 - val_loss: 0.9484
Restoring model weights from the end of the best epoch: 5.


In [30]:
train_loss, train_accuracy = densenet121_model_64.evaluate(train_dataset)
val_loss, val_accuracy = densenet121_model_64.evaluate(val_dataset)
test_loss, test_accuracy = densenet121_model_64.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 180s 2s/step - accuracy: 0.6592 - loss: 0.9757
24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.6398 - loss: 0.9484
24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.6440 - loss: 0.9968


In [31]:
densenet121_model_64.save("models/densenet121_model_64.keras")

In [34]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load DenseNet121

base_model = DenseNet121(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

# Fine-Tuning

base_model.trainable = True

# Freeze all layers except the last 30

for layer in base_model.layers[:-30]:

    layer.trainable = False

# Build Model

DesNet_ft = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        7,
        activation="softmax"
    )

])

DesNet_ft.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 773,511 (2.95 MB)

 Non-trainable params: 6,396,096 (24.40 MB)

In [35]:
DesNet_ft.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_DesNet_ft = DesNet_ft.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 242s 2s/step - accuracy: 0.2496 - loss: 1.9926 - val_accuracy: 0.5506 - val_loss: 1.4330
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 225s 2s/step - accuracy: 0.4394 - loss: 1.5603 - val_accuracy: 0.5945 - val_loss: 1.3077
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.5110 - loss: 1.4097 - val_accuracy: 0.5806 - val_loss: 1.2394
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 492s 4s/step - accuracy: 0.5180 - loss: 1.2611 - val_accuracy: 0.6491 - val_loss: 1.0068
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.5780 - loss: 1.1429 - val_accuracy: 0.6405 - val_loss: 1.0148
Restoring model weights from the end of the best epoch: 4.


In [36]:
train_loss, train_accuracy = DesNet_ft.evaluate(train_dataset)
val_loss, val_accuracy = DesNet_ft.evaluate(val_dataset)
test_loss, test_accuracy = DesNet_ft.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.6525 - loss: 1.0353
24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.6491 - loss: 1.0068
24/24 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step - accuracy: 0.6447 - loss: 1.0465


In [37]:
DesNet_ft.save("models/mobilenet_ft.keras")

In [38]:
import keras_tuner as kt
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, RMSprop

num_classes = 7

def build_model(hp):

    base_model = DenseNet121(

        weights="imagenet",

        include_top=False,

        input_shape=(224,224,3)

    )

    base_model.trainable = False

    DesNet_model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(

            units=hp.Choice(

                "dense_units",

                [128,256,512]

            ),

            activation="relu"

        ),

        layers.Dropout(

            hp.Choice(

                "dropout",

                [0.3,0.5,0.6]

            )

        ),

        layers.Dense(

            num_classes,

            activation="softmax"

        )

    ])

    learning_rate = hp.Choice(

        "learning_rate",

        [1e-3,1e-4,1e-5]

    )

    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]

    )

    if optimizer == "adam":

        opt = Adam(

            learning_rate=learning_rate

        )

    else:

        opt = RMSprop(

            learning_rate=learning_rate

        )

    DesNet_model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return DesNet_model

In [39]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=3,

    directory="DesNet_tuner",

    project_name="mobilenet_hyperparameter"

)

In [40]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights

)

Trial 3 Complete [00h 18m 25s]
val_accuracy: 0.625832200050354

Best val_accuracy So Far: 0.625832200050354
Total elapsed time: 00h 55m 44s


In [41]:
best_DesNet = tuner.get_best_models(1)[0]

/Users/aximsoft/Documents/untitled folder/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [42]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'dense_units': 256, 'dropout': 0.3, 'learning_rate': 0.0001, 'optimizer': 'rmsprop'}


In [43]:

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
desnet_final= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256,activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(7,activation="softmax")

])

desnet_final.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,301,703 (27.85 MB)

 Trainable params: 264,199 (1.01 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [44]:
desnet_final.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_DesNet_final= desnet_final.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 376s 3s/step - accuracy: 0.2016 - loss: 1.8657 - val_accuracy: 0.4115 - val_loss: 1.5955
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 411s 4s/step - accuracy: 0.3726 - loss: 1.6389 - val_accuracy: 0.5939 - val_loss: 1.2935
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 408s 4s/step - accuracy: 0.4472 - loss: 1.5131 - val_accuracy: 0.5486 - val_loss: 1.2854
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 406s 4s/step - accuracy: 0.4748 - loss: 1.4349 - val_accuracy: 0.6312 - val_loss: 1.1137
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 417s 4s/step - accuracy: 0.5284 - loss: 1.3685 - val_accuracy: 0.6039 - val_loss: 1.1513
Restoring model weights from the end of the best epoch: 4.


In [45]:
train_loss, train_accuracy = desnet_final.evaluate(train_dataset)
val_loss, val_accuracy = desnet_final.evaluate(val_dataset)
test_loss, test_accuracy = desnet_final.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 333s 3s/step - accuracy: 0.6138 - loss: 1.1998
24/24 ━━━━━━━━━━━━━━━━━━━━ 71s 3s/step - accuracy: 0.6312 - loss: 1.1137
24/24 ━━━━━━━━━━━━━━━━━━━━ 72s 3s/step - accuracy: 0.6201 - loss: 1.1481


In [46]:
desnet_final.save("models/desnet_final.keras")